# coffee-leaf-ai: training (Kaggle / Colab GPU)

Input: `coffee_leaf_kaggle.zip` made locally by `ml/scripts/04_make_processed.py`
(contains `data/processed/`, `configs/config.yaml`, `ml/src/`).

Steps: load metadata -> DataLoaders -> timm model -> stage 1 (head only) -> stage 2 (full fine-tune)
-> evaluate (val, test, external tests) -> ONNX export -> `artifacts/` with
`model.onnx`, `labels.json`, `preprocess.json`, `metrics.json`.

**Kaggle:** add the uploaded zip as a dataset, turn on GPU, Run All.
**Colab:** Runtime -> GPU, then run the Colab cell below to copy the zip from Drive.

In [ ]:
%pip install -q timm onnx onnxscript onnxruntime

In [ ]:
# Colab only: uncomment to copy the zip from Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# !cp "/content/drive/MyDrive/coffee_leaf_kaggle.zip" /content/

In [ ]:
import glob
import os
import shutil
import sys
import time
import zipfile
from pathlib import Path


def find_project_root() -> Path:
    """Folder containing configs/config.yaml, unpacked from coffee_leaf_kaggle.zip."""
    if os.environ.get("COFFEE_ROOT"):  # local runs / smoke test
        return Path(os.environ["COFFEE_ROOT"])
    hits = glob.glob("/kaggle/input/*/configs/config.yaml") + glob.glob("/kaggle/input/*/*/configs/config.yaml")
    colab_zip = Path("/content/coffee_leaf_kaggle.zip")
    if not hits and colab_zip.exists():
        with zipfile.ZipFile(colab_zip) as zf:
            zf.extractall("/content/coffee-leaf-ai")
        hits = glob.glob("/content/coffee-leaf-ai/configs/config.yaml")
    if not hits:
        raise FileNotFoundError(
            "configs/config.yaml not found. Kaggle: add the dataset made from coffee_leaf_kaggle.zip. "
            "Colab: copy the zip to /content/coffee_leaf_kaggle.zip (cell above).")
    return Path(hits[0]).parents[1]


ROOT = find_project_root()
OUT_DIR = Path(os.environ.get("COFFEE_OUT") or ("/kaggle/working/artifacts" if Path("/kaggle/working").exists()
                                               else ROOT / "artifacts"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "ml"))
print("project root:", ROOT, "| outputs:", OUT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import timm
import torch
from torch import nn
from tqdm.auto import tqdm

from src.common import load_config
from src.dataset import class_weights, load_metadata, make_loader
from src.evaluate import compute_metrics, plot_confusion_matrix, predict, threshold_sweep
from src.export_onnx import export_onnx, verify_onnx, write_artifacts
from src.transforms import eval_transforms, preprocess_spec, train_transforms

cfg = load_config(ROOT / "configs" / "config.yaml")
CLASSES = cfg["classes"]
T = dict(cfg["train"])
PROCESSED = ROOT / cfg["paths"]["processed_dir"]

# COFFEE_SMOKE=1: tiny CPU run used to check this notebook end-to-end, not a real training run.
SMOKE = os.environ.get("COFFEE_SMOKE") == "1"
if SMOKE:
    T.update(pretrained=False, batch_size=4, num_workers=0,
             stage1={**T["stage1"], "epochs": 1}, stage2={**T["stage2"], "epochs": 1})

SEED = cfg["split"]["seed"]
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| model:", T["model_name"], "| classes:", CLASSES)

## Data

In [ ]:
df = load_metadata(PROCESSED, CLASSES)
print(df.pivot_table(index="split", columns="label", values="path", aggfunc="count", fill_value=0)
        .reindex(columns=CLASSES, fill_value=0))

frames = {s: g for s, g in df.groupby("split")}
for required in ("train", "val"):
    assert required in frames, f"split '{required}' missing from metadata.csv"

train_loader = make_loader(frames["train"], PROCESSED, CLASSES, train_transforms(cfg),
                           T["batch_size"], shuffle=True, num_workers=T["num_workers"])
eval_loaders = {s: make_loader(f, PROCESSED, CLASSES, eval_transforms(cfg),
                               T["batch_size"] * 2, shuffle=False, num_workers=T["num_workers"])
                for s, f in frames.items() if s != "train"}
print({s: len(l.dataset) for s, l in {"train": train_loader, **eval_loaders}.items()})

## Model

In [ ]:
model = timm.create_model(T["model_name"], pretrained=T["pretrained"], num_classes=len(CLASSES)).to(DEVICE)
weights = class_weights(frames["train"], CLASSES).to(DEVICE) if T["class_weighted_loss"] else None
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=T["label_smoothing"])
print("class weights:", None if weights is None else weights.tolist())

## Two-stage fine-tune
Checkpoints are saved every epoch: `checkpoints/last.pt` and `checkpoints/best.pt` (by val macro-F1).

In [ ]:
CKPT_DIR = OUT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
USE_AMP = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
history, best = [], {"macro_f1": -1.0}


def set_backbone_trainable(trainable: bool) -> None:
    for p in model.parameters():
        p.requires_grad = trainable
    for p in model.get_classifier().parameters():
        p.requires_grad = True


def train_one_epoch(optimizer, scheduler) -> float:
    model.train()
    total, n = 0.0, 0
    for x, y in tqdm(train_loader, leave=False):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item() * len(y)
        n += len(y)
    return total / n


def run_stage(name: str, epochs: int, lr: float) -> None:
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=T["weight_decay"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, pct_start=0.1,
                                                    total_steps=epochs * len(train_loader))
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        loss = train_one_epoch(optimizer, scheduler)
        y, p = predict(model, eval_loaders["val"], DEVICE)
        m = compute_metrics(y, p, CLASSES)
        row = {"stage": name, "epoch": epoch, "train_loss": round(loss, 4),
               "val_acc": round(m["accuracy"], 4), "val_macro_f1": round(m["macro_f1"], 4),
               "sec": round(time.time() - t0)}
        history.append(row)
        print(row)
        state = {"model": model.state_dict(), **row}
        torch.save(state, CKPT_DIR / "last.pt")
        if m["macro_f1"] > best["macro_f1"]:
            best.update(macro_f1=m["macro_f1"], stage=name, epoch=epoch)
            torch.save(state, CKPT_DIR / "best.pt")
    # TODO: early stopping, and resuming from last.pt if a Kaggle/Colab session times out.

In [ ]:
# Stage 1: backbone frozen, train the new classifier head.
set_backbone_trainable(False)
run_stage("stage1_head", T["stage1"]["epochs"], T["stage1"]["lr"])

# Stage 2: unfreeze everything, lower learning rate.
set_backbone_trainable(True)
run_stage("stage2_full", T["stage2"]["epochs"], T["stage2"]["lr"])
print("best:", best)

## Evaluation
`test` is in-domain (jmuben + uganda). `test_bracol` / `test_rocole` are external; rocole only has healthy/rust, so its macro-F1 covers those two classes.

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / "best.pt", map_location=DEVICE)["model"])
metrics = {"model_name": T["model_name"], "classes": CLASSES, "best": best, "history": history, "splits": {}}
for split, loader in eval_loaders.items():
    y, p = predict(model, loader, DEVICE)
    m = compute_metrics(y, p, CLASSES)
    metrics["splits"][split] = m
    plot_confusion_matrix(m["confusion_matrix"], CLASSES, OUT_DIR / f"cm_{split}.png", title=split)
    if split == "val":
        metrics["val_threshold_sweep"] = threshold_sweep(y, p)
    print(f"{split:>13}: n={m['n']:>6}  acc={m['accuracy']:.3f}  macro_f1={m['macro_f1']:.3f}")

# TODO: pick inference.confidence_threshold in configs/config.yaml from this table
# (e.g. the lowest threshold whose accuracy_on_accepted meets your target).
pd.DataFrame(metrics["val_threshold_sweep"])

## Export to ONNX

In [ ]:
onnx_path = export_onnx(model, OUT_DIR / "model.onnx", cfg["image"]["size"], T["onnx_opset"])
print("relative |torch - onnx| =", verify_onnx(model, onnx_path, cfg["image"]["size"]))
write_artifacts(OUT_DIR, CLASSES, preprocess_spec(cfg), metrics)

# Small zip for download (checkpoints are left out).
bundle = OUT_DIR.parent / "coffee_leaf_model.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["model.onnx", "labels.json", "preprocess.json", "metrics.json", *[p.name for p in OUT_DIR.glob("cm_*.png")]]:
        zf.write(OUT_DIR / f, f)
print("download:", bundle)

## Next
Download `coffee_leaf_model.zip` (Kaggle: Output tab; Colab: Files panel), unzip it into
`backend/models/` locally so it contains `model.onnx`, `labels.json`, `preprocess.json`, and restart the backend.